# SQL Murder Mystery
A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a murder that occurred sometime on **Jan.15, 2018** and that it took place in SQL City. All the clues to this mystery are buried in a huge database, and you need to use SQL to navigate through this vast network of information. Your first step to solving the mystery is to retrieve the corresponding crime scene report from the police department’s database. Take a look at the cheatsheet to learn how to do this! From there, you can use your SQL skills to find the murderer.

## Exploring tables, see the schema below
![schema.png](schema.png)

#### interview:

In [4]:
%%sql
SELECT * FROM interview LIMIT 5

person_id,transcript
28508,‘I deny it!’ said the March Hare.
63713,
86208,"way, and the whole party swam to the shore."
35267,"lessons in here? Why, there’s hardly room for YOU, and no room at all"
33856,


#### get_fit_now_check_in:

In [128]:
%%sql
SELECT * FROM get_fit_now_check_in LIMIT 5;

membership_id,check_in_date,check_in_time,check_out_time
NL318,20180212,329,365
NL318,20170811,469,920
NL318,20180429,506,554
NL318,20180128,124,759
NL318,20171027,418,1019


#### get_fit_now_member:

In [129]:
%%sql
SELECT * FROM get_fit_now_member LIMIT 5;

id,person_id,name,membership_start_date,membership_status
NL318,65076,Everette Koepke,20170926,gold
AOE21,39426,Noe Locascio,20171005,regular
2PN28,63823,Jeromy Heitschmidt,20180215,silver
0YJ24,80651,Waneta Wellard,20171206,gold
3A08L,32858,Mei Bianchin,20170401,silver


#### facebook_event_checkin:

In [130]:
%%sql
SELECT * FROM facebook_event_checkin LIMIT 5;

person_id,event_id,event_name,date
28508,5880,Nudists are people who wear one-button suits.,20170913
63713,3865,but that's because it's the best book on anything for the layman.,20171009
63713,3999,"If Murphy's Law can go wrong, it will.",20170502
63713,6436,Old programmers never die. They just branch to a new address.,20170926
82998,4470,Help a swallow land at Capistrano.,20171022


#### person:

In [131]:
%%sql
SELECT * FROM person LIMIT 5;

id,name,license_id,address_number,address_street_name,ssn
10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
10010,Muoi Cary,385336,741,Northwestern Dr,828638512
10016,Era Moselle,431897,1987,Wood Glade St,614621061
10025,Trena Hornby,550890,276,Daws Hill Way,223877684


#### drivers_license:

In [132]:
%%sql
SELECT * FROM drivers_license LIMIT 5;

id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
100280,72,57,brown,red,male,P24L4U,Acura,MDX
100460,63,72,brown,brown,female,XF02T6,Cadillac,SRX
101029,62,74,green,green,female,VKY5KR,Scion,xB
101198,43,54,amber,brown,female,Y5NZ08,Nissan,Rogue
101255,18,79,blue,grey,female,5162Z1,Lexus,GS


#### income:

In [133]:
%%sql
SELECT * FROM income LIMIT 5;

ssn,annual_income
100009868,52200
100169584,64500
100300433,74400
100355733,35900
100366269,73000


#### crime_scene_report:

In [134]:
%%sql
SELECT * FROM crime_scene_report LIMIT 5;

date,type,description,city
20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
20180115,murder,Life? Dont talk to me about life.,Albany
20180115,murder,"Mama, I killed a man, put a gun against his head...",Reno
20180215,murder,REDACTED REDACTED REDACTED,SQL City
20180215,murder,Someone killed the guard! He took an arrow to the knee!,SQL City


#### Check the format of the dates and extract useful reports from `crime_scene_report`:

In [135]:
%%sql
SELECT  date,
        TO_DATE(CAST(date AS VARCHAR ), 'YYYYMMDD') AS date_check,
        type,
        description,
        city
FROM crime_scene_report
WHERE type = 'murder'
AND city = 'SQL City'
AND date = 20180115;


date,date_check,type,description,city
20180115,2018-01-15,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


**Description of the event:**

_Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave"._

#### Create `view` named `witnesses` with information about the witnesses:

In [136]:
%%sql
-- drop already created views to run this notebook codes again
DROP VIEW IF EXISTS suspect_check_ins;
DROP VIEW IF EXISTS alice_gym_time;
DROP VIEW IF EXISTS witnesses;

[]

In [137]:
%%sql
CREATE VIEW witnesses AS
SELECT *
FROM person
WHERE (address_street_name = 'Northwestern Dr'
        AND address_number >= ALL(SELECT address_number FROM person))
OR (address_street_name = 'Franklin Ave' AND name LIKE 'Annabel%');

[]

In [138]:
%%sql
SELECT * FROM witnesses;

id,name,license_id,address_number,address_street_name,ssn
14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
16371,Annabel Miller,490173,103,Franklin Ave,318771143


#### Check the interviews with those people:

In [139]:
%%sql
SELECT witnesses.id AS person_id,
    name,
    transcript
FROM witnesses LEFT JOIN interview ON witnesses.id = interview.person_id;

person_id,name,transcript
14887,Morty Schapiro,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
16371,Annabel Miller,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


#### Clues:
- **Morty Schapiro:**
_I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W"_
- **Annabel Miller:**
_I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th._

Create `view` named `alice_gym_time` to find out times at which Alice was in the gym and use them later:

In [140]:
%%sql
CREATE VIEW alice_gym_time AS
SELECT membership_id,
       check_in_date,
       check_in_time,
       check_out_time
FROM get_fit_now_check_in JOIN get_fit_now_member ON membership_id = id
WHERE check_in_date = 20180109
AND membership_id IN
    (SELECT get_fit_now_member.id
    FROM witnesses JOIN get_fit_now_member ON witnesses.id = get_fit_now_member.person_id
    WHERE witnesses.id = 16371);

[]

In [141]:
%%sql
SELECT * FROM alice_gym_time;

membership_id,check_in_date,check_in_time,check_out_time
90081,20180109,1600,1700


#### Create `view` named `suspect_check_ins` to list membership ids beginning with '48Z' of people who were in the gym at the same time as Alice:

In [142]:
%%sql
CREATE VIEW suspect_check_ins AS
SELECT *
FROM get_fit_now_check_in
WHERE check_in_date = 20180109
AND check_in_time + check_out_time BETWEEN
        (SELECT (check_in_time + check_out_time) - (check_out_time - check_in_time) FROM alice_gym_time)
         AND
        (SELECT (check_in_time + check_out_time) + (check_out_time - check_in_time) FROM alice_gym_time)
AND membership_id <> (SELECT membership_id FROM alice_gym_time)
AND membership_id LIKE '48Z%';

[]

In [143]:
%%sql
SELECT * FROM suspect_check_ins;

membership_id,check_in_date,check_in_time,check_out_time
48Z7A,20180109,1600,1730
48Z55,20180109,1530,1700


#### Identify the name of the murderer based on the membership ids from `suspect_check_ins` and its plate number containing 'H42'

In [144]:
%%sql
SELECT person.name
FROM person JOIN drivers_license ON license_id = drivers_license.id
WHERE plate_number LIKE '%H42W%'
AND person.id IN
    (SELECT person_id
     FROM suspect_check_ins JOIN get_fit_now_member ON membership_id = id
     WHERE membership_id = get_fit_now_member.id)


name
Jeremy Bowers


#### The solution checker confirms 'Jeremy Bowers' as the murderer, but there is supposedly someone, who is the big brain of this crime!

In [145]:
%%sql
SELECT person_id,
       name,
       transcript
FROM interview JOIN person ON person_id = id
WHERE name = 'Jeremy Bowers'

person_id,name,transcript
67318,Jeremy Bowers,"I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5"" (65"") or 5'7"" (67""). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017."


Find the brain behind the murder based on this information from interview:

_I was hired by a woman with a lot of money. I don't know her name, but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017._


In [146]:
%%sql
SELECT person.name,
       height,
       hair_color,
       car_make,
       car_model,
       event_name,
       annual_income,
       COUNT(date) AS number_of_attendences
FROM person JOIN drivers_license ON license_id = drivers_license.id
            JOIN facebook_event_checkin ON person.id = person_id
            JOIN income ON person.ssn = income.ssn
WHERE height BETWEEN 65 AND 67
AND hair_color = 'red'
AND car_make = 'Tesla'
GROUP BY person.name, height, hair_color, car_make, car_model, event_name, annual_income;


name,height,hair_color,car_make,car_model,event_name,annual_income,number_of_attendences
Miranda Priestly,66,red,Tesla,Model S,SQL Symphony Concert,310000,3


#### The solution checker confirms 'Miranda Priestly' as the criminal behind the murder.

## Case closed!